In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"]=(20,10)
import seaborn as sns;sns.set()
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
from datetime import datetime,time
from plotly.offline import iplot,plot,init_notebook_mode,download_plotlyjs
%matplotlib inline 
init_notebook_mode(connected=True)

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm, norm, gamma

def price_conditional_distribution_prob_all(
    df, price_col, threshold, category_col=None, dist="lognormal"
):
    """
    Compute all conditional probabilities using a fitted distribution.
    Calculates: P(>), P(<), P(=), P(>=), P(<=)
    Category column (if provided) is shown first in the output.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing price column and optional categorical column.
    price_col : str
        Numeric column to fit distribution on.
    threshold : float
        Threshold value for probability calculation.
    category_col : str, optional
        Categorical column name to condition on.
    dist : str
        Distribution type: 'lognormal', 'normal', or 'gamma'.

    Returns
    -------
    pd.DataFrame
        Probabilities per category (or overall if no category).
    """

    valid_dists = ["lognormal", "normal", "gamma"]
    if dist not in valid_dists:
        raise ValueError(f"Invalid distribution '{dist}'. Must be one of {valid_dists}")

    results = []

    # --- helper function ---
    def compute_probs(prices):
        # Fit the chosen distribution
        if dist == "lognormal":
            shape, loc, scale = lognorm.fit(prices, floc=0)
            cdf_val = lognorm.cdf(threshold, shape, loc=loc, scale=scale)
        elif dist == "normal":
            mu, sigma = norm.fit(prices)
            cdf_val = norm.cdf(threshold, mu, sigma)
        elif dist == "gamma":
            shape, loc, scale = gamma.fit(prices, floc=0)
            cdf_val = gamma.cdf(threshold, shape, loc=loc, scale=scale)

        # Convert to probabilities for all comparison operators
        probs = {
            f"P({price_col} > {threshold})": 1 - cdf_val,
            f"P({price_col} < {threshold})": cdf_val,
            f"P({price_col} >= {threshold})": 1 - cdf_val,
            f"P({price_col} <= {threshold})": cdf_val,
            f"P({price_col} = {threshold})": 0.0,  # continuous dist → 0
        }
        return probs

    # --- no categorical condition ---
    if category_col is None:
        prices = df[price_col]
        probs = compute_probs(prices)
        # Make category column for consistency
        probs = {"Condition": "All", **probs}
        results.append(probs)
        df_result = pd.DataFrame(results)
        df_result = df_result[
            ["Condition"]
            + [col for col in df_result.columns if col != "Condition"]
        ]
        return df_result

    # --- with categorical condition ---
    for cat, group in df.groupby(category_col):
        prices = group[price_col]
        probs = compute_probs(prices)
        # Make sure category_col is first
        probs = {category_col: cat, **probs}
        results.append(probs)

    df_result = pd.DataFrame(results)
    df_result = df_result[
        [category_col]
        + [col for col in df_result.columns if col != category_col]
    ]
    return df_result


In [4]:
import numpy as np
import pandas as pd

def price_conditional_empirical_prob_all(df, price_col, threshold, category_col=None):
    """
    Compute all empirical conditional probabilities for a numeric variable:
    P(>), P(<), P(=), P(>=), P(<=)
    
    Optionally conditioned on a categorical column.
    The category column (if provided) appears first in the output.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing the data.
    price_col : str
        Column name of the numeric variable (e.g. 'price').
    threshold : float or int
        Threshold value for the probability calculation.
    category_col : str, optional
        Categorical column name for grouping (e.g. 'segment').

    Returns
    -------
    pd.DataFrame
        A DataFrame with empirical probabilities (by group if provided).
    """
    results = []

    # Helper function to compute all probabilities for a numeric Series
    def compute_probs(values):
        probs = {
            f"P({price_col} > {threshold})": np.mean(values > threshold),
            f"P({price_col} < {threshold})": np.mean(values < threshold),
            f"P({price_col} = {threshold})": np.mean(values == threshold),
            f"P({price_col} >= {threshold})": np.mean(values >= threshold),
            f"P({price_col} <= {threshold})": np.mean(values <= threshold),
        }
        return probs

    # No category condition → overall probabilities
    if category_col is None:
        prices = df[price_col]
        probs = compute_probs(prices)
        probs = {"Condition": "All", **probs}
        results.append(probs)
        df_result = pd.DataFrame(results)
        df_result = df_result[
            ["Condition"]
            + [col for col in df_result.columns if col != "Condition"]
        ]
        return df_result

    # With category condition → probabilities per group
    for cat, group in df.groupby(category_col):
        prices = group[price_col]
        probs = compute_probs(prices)
        probs = {category_col: cat, **probs}
        results.append(probs)

    df_result = pd.DataFrame(results)
    df_result = df_result[
        [category_col]
        + [col for col in df_result.columns if col != category_col]
    ]
    return df_result


In [5]:
from scipy.stats import spearmanr,mannwhitneyu
def mannwhitneyu_func(Dataset:pd.DataFrame,Numericaltarget:str,BinaryFeature:str):
    group1 = Dataset[Numericaltarget][Dataset[BinaryFeature] == 0]
    group2 = Dataset[Numericaltarget][Dataset[BinaryFeature]  == 1]
    u_stat, p_mw = mannwhitneyu(group1, group2, alternative='two-sided')
    print(f"W = {u_stat:.3f}, p = {p_mw:.3f}")
    if p_mw < 0.05:
        print(f"As The P-value is less than .05 : Reject H₀ → There is significant distribution difference between {Numericaltarget} & {BinaryFeature}")
    else:
        print(f"As The P-value is more than .05 : Fail to reject H₀ → No significant distribution difference between {Numericaltarget} & {BinaryFeature}")

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    sns.boxplot(x=BinaryFeature, y=Numericaltarget, data=Dataset, palette="Set2",ax=axes[0])
    sns.stripplot(x=BinaryFeature, y=Numericaltarget, data=Dataset, color="black", alpha=0.6,ax=axes[1]);
    sns.histplot(x=Numericaltarget,hue=BinaryFeature,ax=axes[2],data=Dataset,alpha=0.6)

In [6]:
def spearmanr_func(Dataset:pd.DataFrame,Numericaltarget:str,OrdinalFeature:str):
    spearman_corr, p_spear = spearmanr(Dataset[Numericaltarget],Dataset[OrdinalFeature])
    prob_df = Dataset.groupby(OrdinalFeature)[Numericaltarget].mean().reset_index()
    prob_df.rename(columns={Numericaltarget: 'prob_target_1'}, inplace=True)  

    print(f"W = {spearman_corr:.3f}, p = {p_spear:.3f}")
    if p_spear < 0.05:
        print(f"As The P-value is less than .05 : Reject H₀ → There is significant monotonic relationship between {Numericaltarget} & {OrdinalFeature}")
    else:
        print(f"As The P-value is more than .05 : Fail to reject H₀ → No significant monotonic relationship between {Numericaltarget} & {OrdinalFeature}")

    plt.figure(figsize=(20,5))
    ax=sns.heatmap(prob_df.set_index(OrdinalFeature).T, annot=True, cmap="YlGnBu", cbar=True)
    ax.set_title("Probability of Binary Target=1 by Ordinal Feature")
    ax.set_xlabel("Ordinal Feature")
    ax.set_ylabel("Probability")
    plt.tight_layout()
    plt.show()  

In [7]:
def kruskal_func(Dataset:pd.DataFrame,Numericaltarget:str,CategoricalFeature:str):
    from scipy.stats import kruskal
    groups = [Dataset[Dataset[CategoricalFeature] == level][Numericaltarget] for level in Dataset[CategoricalFeature].unique()]

    stat, p = kruskal(*groups)
    if p < 0.05: 
        print(f"Kruskal-Wallis Test: Statistic={stat}, P-value={p} ==>\n The p value less than 0.05 ==> At least one group has Significant difference")
    else:
        print(f"Kruskal-Wallis Test: Statistic={stat}, P-value={p} ==>\n the p value more than 0.05 ==> No Significant difference between groups")

    fig, axes = plt.subplots(1, 2, figsize=(20, 5))
    sns.boxplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, palette="Set2",ax=axes[0])
    sns.stripplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, color="black", alpha=0.6, jitter=True,ax=axes[1])
    plt.title(f"Kruskal–Wallis Test\nH={stat:.2f}, p={p:.4f}")
    plt.show()

    if p < 0.05:
        print("As At least one group has Significant difference --> posthoc_dunn test")
        import scikit_posthocs as sp
        posthoc = sp.posthoc_dunn(Dataset, val_col='price', group_col='p_color', p_adjust='bonferroni')
        print(posthoc)
        plt.figure(figsize=(20,5))
        sns.heatmap(posthoc,annot=True,cmap="Reds", cbar=False, linewidths=0.5, linecolor='gray',fmt=".2f");